# 🛡️ ISO 27001 Compliance Checker (Ollama + RAG)

Takes the **company document (PDF)** and the **ISO 27001 requirements (CSV)**, and produces a report that shows for each requirement:
- ✅ Compliant / 🟡 Partially compliant / ❌ Not compliant
- **Evidence** (text from the company document + page number)
- **Recommended fix** for anything that is not compliant

**Before running** (in a terminal):
```
ollama pull llama3:8b
ollama serve
```

## 1. Configuration

In [ ]:
import re
import html
import fitz
import requests
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

ISO_CSV      = "C:\\Users\\sasap\\OneDrive\\Desktop\\Task2\\iso-27001-annex-a.csv"
COMPANY_PDF  = "C:\\Users\\sasap\\OneDrive\\Desktop\\Task2\\Company_ISO27001_Evidence_Sample.pdf"
REPORT_CSV   = "iso27001_compliance_report.csv"
REPORT_HTML  = "iso27001_compliance_report.html"

MODEL       = "llama3:8b"
OLLAMA_URL  = "http://localhost:11434/api/generate"

CHUNK_SIZE = 300
OVERLAP    = 80
MIN_EVIDENCE_SCORE = 0.35

embedder = SentenceTransformer("all-MiniLM-L6-v2")

try:
    names = [m["name"] for m in requests.get("http://localhost:11434/api/tags", timeout=5).json().get("models", [])]
    print("Ollama OK. Models:", names)
    
    if not any(n.startswith(MODEL) for n in names):
        print(f"Model not found, run: ollama pull {MODEL}")
        
except Exception as e:
    print("Ollama is not running. Start it with: ollama serve\n", e)

## 2. Read the company PDF and build the index

In [ ]:
def normalize_text(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()


def keyword_score(query, text):
    q = set(normalize_text(query).split())
    t = set(normalize_text(text).split())
    return len(q & t) / len(q) if q else 0.0


def build_index(pdf_path):
    chunks = []
    for page_no, page in enumerate(fitz.open(pdf_path), start=1):
        words = page.get_text("text").split()
        
        for start in range(0, len(words), CHUNK_SIZE - OVERLAP):
            part = words[start:start + CHUNK_SIZE]
            
            if len(part) >= 10:
                chunks.append({"page": page_no, "text": " ".join(part)})

    if not chunks:
        raise ValueError(f"No selectable text in {pdf_path} (scanned PDF? needs OCR).")

    emb = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=True)
    
    return {"chunks": chunks, "emb": emb}


def search_index(index, query, k=4):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    sem = index["emb"] @ q
    kw = np.array([keyword_score(query, c["text"]) for c in index["chunks"]])
    score = 0.75 * sem + 0.25 * kw
    
    return [{**index["chunks"][i], "score": float(score[i])} for i in np.argsort(score)[::-1][:k]]


company_index = build_index(COMPANY_PDF)
print("Company chunks:", len(company_index["chunks"]))

## 3. Load the ISO 27001 requirements from the CSV
Columns are detected automatically. If the wrong ones are picked, set `ID_COL / TITLE_COL / TEXT_COL` manually.

In [ ]:
def load_requirements(iso_df, id_col=None, title_col=None, text_col=None):
    iso_df = iso_df.fillna("")
    iso_df.columns = [str(c).strip() for c in iso_df.columns]

    def pick(candidates, exclude=()):
        for key in candidates:
            for col in iso_df.columns:
                if col not in exclude and key in col.lower():
                    return col
        return None

    id_col = id_col or pick(["clause", "control id", "id", "ref", "no", "number"]) or iso_df.columns[0]
    
    title_col = title_col or pick(["title", "name", "control", "requirement"], exclude=(id_col,))
    
    text_col = text_col or pick(["description", "requirement", "statement", "objective", "text", "purpose"],exclude=(id_col, title_col))

    reqs = []
    
    for _, row in iso_df.iterrows():
        
        req_id = str(row[id_col]).strip()
        
        title = str(row[title_col]).strip() if title_col else ""
        
        ref_text = (str(row[text_col]).strip() if text_col
                    
        else " ".join(str(v) for k, v in row.items() if k != id_col))
        
        if not (title or ref_text):
            continue
        
        query = f"{title}. {' '.join(ref_text.split()[:60])}".strip()
        
        reqs.append((req_id, title or req_id, query, ref_text))

    if not reqs:
        raise ValueError("No requirements found in the CSV.")
    
    return reqs, (id_col, title_col, text_col)


iso_df = pd.read_csv(ISO_CSV, encoding="utf-8-sig")

print("CSV columns:", list(iso_df.columns), "| Rows:", len(iso_df))

ID_COL = TITLE_COL = None
TEXT_COL = None

REQUIREMENTS, (ID_COL, TITLE_COL, TEXT_COL) = load_requirements(iso_df, ID_COL, TITLE_COL, TEXT_COL)

print("Using -> ID:", ID_COL, "| TITLE:", TITLE_COL, "| TEXT:", TEXT_COL)

print("Requirements:", len(REQUIREMENTS))

## 4. Check a requirement (company evidence → verdict → recommended fix)

In [ ]:
def call_llm(prompt, max_tokens=150):
    r = requests.post(
        OLLAMA_URL,
        json={"model": MODEL, "prompt": prompt, "stream": False,"options": {"temperature": 0, "num_predict": max_tokens, "num_ctx": 8192}},
        timeout=600,
    )
    
    if not r.ok:
        raise RuntimeError(f"Ollama error {r.status_code}:\n{r.text}")
    return r.json().get("response", "").strip()


def parse_verdict(text):
    m = re.search(r"VERDICT\s*:\s*\**\s*(NOT[_ \-]?COMPLIANT|NON[_ \-]?COMPLIANT|PARTIAL|COMPLIANT)", text, re.I)
    if m:
        v = m.group(1).upper()
        verdict = "NOT_COMPLIANT" if v.startswith(("NOT", "NON")) else v
    else:
        verdict = "PARTIAL"

    r = re.search(r"REASON\s*:\s*(.+)", text, re.I | re.S)
    reason = r.group(1).strip().split("\n")[0] if r else text.strip()[:200]
    return verdict, reason


def suggest_fix(req_id, title, ref_text, gap):
    prompt = f"""
You are an ISO 27001 consultant.

Requirement: {req_id} - {title}
ISO 27001 requirement text:
{ref_text}

Gap found in the company document: {gap}

Give 2 to 3 short, concrete actions the company should take to become compliant.
Write them as a numbered list. No intro, no extra text.
"""
    return call_llm(prompt, max_tokens=200)


def check_requirement(req_id, title, query, ref_text):
    evidence = search_index(company_index, query, k=3)
    
    
    if not evidence:
        return {
            **{"ID": req_id, "Requirement": title},
            "Verdict": "NOT_COMPLIANT",
            "Reason": "No matching evidence found in the index.",
            "Page": None,
            "Evidence": "",
            "Recommendation": suggest_fix(req_id, title, ref_text, "No evidence found"),
            "Score": 0.0,
        }

    best = evidence[0]
    base = {"ID": req_id, "Requirement": title}

    if best["score"] < MIN_EVIDENCE_SCORE:
        verdict = "NOT_COMPLIANT"
        reason = "No relevant evidence found in the company document."
        page = None
        evidence_text = ""
        score = round(best["score"], 3)
        
    
    elif best["score"] >= 0.55:
        verdict = "COMPLIANT"
        reason = "The company document clearly and explicitly covers this requirement."
        page = best["page"]
        evidence_text = best["text"]
        score = round(best["score"], 3)
        
    else:
        context = "\n\n".join(
            f"[Company evidence {i} | Page {e['page']}]\n{e['text']}"
            for i, e in enumerate(evidence, 1)
        )

        prompt = f"""
You are an expert ISO 27001 Lead Auditor. Evaluate if the company evidence satisfies the ISO requirement.

- Requirement ID: 
{req_id}
- Title: 
{title}

ISO 27001 Requirement:
{ref_text}

Company Document Evidence:
{context}

Evaluation Instructions:
1. Be fair and realistic. Look for policies, procedures, responsibilities, or rules in the company evidence.
2. COMPLIANT: The evidence clearly addresses the core objective and rules of the requirement.
3. PARTIAL: The evidence mentions the topic or role, but lacks specific operational procedures, timelines, or owners.
4. NOT_COMPLIANT: There is no relevant mention of this requirement in the evidence.

Answer in exactly this format and nothing else:
VERDICT: <COMPLIANT or PARTIAL or NOT_COMPLIANT>
REASON: <one short sentence explaining why>
"""

        verdict, reason = parse_verdict(call_llm(prompt, max_tokens=120))
        page = best["page"]
        evidence_text = best["text"]
        score = round(best["score"], 3)

        if verdict == "NOT_COMPLIANT":
            page = None
            evidence_text = ""

    fix = suggest_fix(req_id, title, ref_text, reason) if verdict != "COMPLIANT" else ""

    return {
        **base,
        "Verdict": verdict,
        "Reason": reason,
        "Page": page,
        "Evidence": evidence_text,
        "Recommendation": fix,
        "Score": score,
    }

## 5. Run the full compliance check

In [ ]:
results = []
for req in REQUIREMENTS:
    res = check_requirement(*req)
    results.append(res)
    print(f"{res['ID']:8} {res['Verdict']:14} {res['Requirement']}")

df = pd.DataFrame(results)
df["Page"] = df["Page"].astype("Int64")
points = {"COMPLIANT": 1.0, "PARTIAL": 0.5, "NOT_COMPLIANT": 0.0, "UNKNOWN": 0.0}
compliance_pct = 100 * df["Verdict"].map(points).mean()

print("\n" + "=" * 60)
print(f"Overall compliance: {compliance_pct:.1f}%")
print(df["Verdict"].value_counts().to_string())
print("=" * 60)

## 6. Final report (CSV + HTML)

In [ ]:
STYLE = {
    "COMPLIANT":     ("✅ مطابق",       "#1a7f37"),
    "PARTIAL":       ("🟡 مطابق جزئياً", "#9a6700"),
    "NOT_COMPLIANT": ("❌ غير مطابق",   "#cf222e"),
    "UNKNOWN":       ("❓ غير محدد",    "#6e7781"),
}
ORDER = ["NOT_COMPLIANT", "PARTIAL", "UNKNOWN", "COMPLIANT"]


def esc(x):
    return html.escape(str(x))


def build_html(df, pct):
    counts = df["Verdict"].value_counts()
    summary = "".join(
        f'<span class="pill" style="background:{STYLE[v][1]}">{STYLE[v][0]}: {counts.get(v, 0)}</span>'
        for v in ORDER
    )
    body = ""
    for v in ORDER:
        part = df[df["Verdict"] == v]
        if part.empty:
            continue
        label, color = STYLE[v]
        body += f'<h2 style="color:{color}">{label} ({len(part)})</h2>'
        for _, r in part.iterrows():
            ev = ""
            if r["Evidence"]:
                ev = f'<p><b>الدليل (صفحة {esc(r["Page"])}):</b></p><blockquote>{esc(r["Evidence"][:600])}...</blockquote>'
            fix = f'<p><b>الحل المقترح:</b></p><pre>{esc(r["Recommendation"])}</pre>' if r["Recommendation"] else ""
            body += (f'<div class="card" style="border-right:5px solid {color}">'
                     f'<h3>{esc(r["ID"])} — {esc(r["Requirement"])}</h3>'
                     f'<p><b>السبب:</b> {esc(r["Reason"])}</p>{ev}{fix}</div>')

    return f"""<!DOCTYPE html><html dir="rtl" lang="ar"><head><meta charset="utf-8">
<title>ISO 27001 Compliance Report</title>
<style>
body{{font-family:Tahoma,Arial,sans-serif;max-width:900px;margin:30px auto;padding:0 16px;background:#f6f8fa;color:#24292f}}
.pill{{color:#fff;padding:4px 12px;border-radius:14px;margin:4px;display:inline-block}}
.card{{background:#fff;padding:8px 16px;margin:10px 0;border-radius:6px}}
blockquote{{background:#f0f2f4;margin:0;padding:8px 12px;direction:ltr;text-align:left;font-size:13px}}
pre{{white-space:pre-wrap;font-family:inherit;background:#eef6ff;padding:8px 12px;direction:ltr;text-align:left}}
</style></head><body>
<h1>ISO 27001 Compliance Report</h1>
<h2>نسبة المطابقة الكلية: {pct:.1f}%</h2>
<div>{summary}</div>{body}</body></html>"""


df.to_csv(REPORT_CSV, index=False, encoding="utf-8-sig")
with open(REPORT_HTML, "w", encoding="utf-8") as f:
    f.write(build_html(df, compliance_pct))

print("Saved:", REPORT_CSV, "+", REPORT_HTML)
df[["ID", "Requirement", "Verdict", "Reason", "Page", "Recommendation"]]